# Step 6 - The LLM in the evaluation



In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json, re, random, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
import torchvision.transforms as T
from sklearn.metrics import roc_auc_score, cohen_kappa_score
from scipy.stats import spearmanr

BASE = "/content/drive/MyDrive/COMP8240_PatchCore"
DATA = f"{BASE}/tealights_mvtec/tealight"       # photos + my masks
GEM_MASKS = f"{BASE}/tealights_mvtec/gemini_masks"
PC_DIR = f"{BASE}/step5_tealights"
OUT_DIR = f"{BASE}/step6_llm_judge"
UP = f"{OUT_DIR}/upload_to_gemini"              # only this folder gets uploaded
os.makedirs(OUT_DIR, exist_ok=True)

# PatchCore outputs from step 5
sc = pd.read_csv(f"{PC_DIR}/tealight_scores.csv")
RUN = sorted(glob.glob(f"{PC_DIR}/raw_results/WR50_tealight*"))[-1]
seg = np.load(glob.glob(f"{RUN}/segmentations_mvtec_tealight.npy")[0])
assert len(seg) == len(sc)
print(sc["type"].value_counts().to_dict())

prep = T.Compose([T.Resize(256), T.CenterCrop(224)])   # what PatchCore actually saw
def load_mask(path):
    return np.array(prep(Image.open(path).convert("L"))) > 127

Mounted at /content/drive
{'used': 49, 'empty': 46, 'good': 18}


# Part 1: PatchCore scored against my masks and against Gemini's masks


In [2]:
me, gem, missing = [], [], []
for i, r in sc.iterrows():
    if r["type"] == "good":
        m1 = m2 = np.zeros((224, 224), bool)
    else:
        m1 = load_mask(f"{DATA}/ground_truth/{r['type']}/{r['name']}_mask.png")
        gpath = f"{GEM_MASKS}/{r['name']}_mask.png"
        if os.path.exists(gpath):
            m2 = load_mask(gpath)
        else:
            m2 = np.zeros((224, 224), bool); missing.append(r["name"])
    me.append(m1); gem.append(m2)
me, gem = np.array(me), np.array(gem)
print("defect images without a Gemini mask:", missing)

rows = []
for label, types in [("all", ["good", "used", "empty"]), ("used", ["good", "used"]), ("empty", ["good", "empty"])]:
    k = sc.type.isin(types).values
    rows.append({"test set": label,
                 "pixel AUROC, my masks": round(100 * roc_auc_score(me[k].ravel(), seg[k].ravel()), 1),
                 "pixel AUROC, Gemini's masks": round(100 * roc_auc_score(gem[k].ravel(), seg[k].ravel()), 1)})
part1 = pd.DataFrame(rows)
part1["difference"] = (part1["pixel AUROC, Gemini's masks"] - part1["pixel AUROC, my masks"]).round(1)
display(part1)
part1.to_csv(f"{OUT_DIR}/pixel_auroc_me_vs_gemini_masks.csv", index=False)

defect images without a Gemini mask: []


,test set,"pixel AUROC, my masks","pixel AUROC, Gemini's masks",difference
0,all,96.9,93.0,-3.9
1,used,92.3,94.9,2.6
2,empty,99.6,94.0,-5.6


# Part 2: Gemini as a judge of PatchCore's anomaly maps


In [3]:
try:
    FONT = ImageFont.load_default(size=26)
except TypeError:
    FONT = ImageFont.load_default()

def overlay(img224, amap):
    # red/yellow = high anomaly score, blue = low
    base = np.asarray(img224, dtype=float) / 255
    heat = plt.get_cmap("jet")(np.clip(amap, 0, 1))[..., :3]
    return Image.fromarray(((0.55 * base + 0.45 * heat) * 255).astype(np.uint8))

def panel(images, labels, w=300, top=34):
    canvas = Image.new("RGB", (len(images) * (w + 12) - 12, w + top), "white")
    d = ImageDraw.Draw(canvas)
    for i, (im, lab) in enumerate(zip(images, labels)):
        x = i * (w + 12)
        canvas.paste(im.convert("RGB").resize((w, w)), (x, top))
        d.text((x + 6, 4), lab, fill="black", font=FONT)
    return canvas

shutil.rmtree(UP, ignore_errors=True); os.makedirs(UP)
random.seed(2)
idx_B = random.sample(sc.index[sc.type != "good"].tolist(), 20)
key_B = pd.DataFrame({"id": [f"B{i + 1:02d}" for i in range(20)], "row": idx_B})
for r in key_B.itertuples():
    row = sc.loc[r.row]
    img = prep(Image.open(f"{DATA}/test/{row['type']}/{row['name']}.png").convert("RGB"))
    mask = Image.fromarray((me[r.row] * 255).astype(np.uint8))
    panel([img, mask, overlay(img, seg[r.row])],
          [f"{r.id}  TEST IMAGE", "REFERENCE", "DETECTOR"]).save(f"{UP}/{r.id}.png")
key_B.to_csv(f"{OUT_DIR}/key_B.csv", index=False)   # kept OUT of the upload folder

PROMPT_B = '''You are evaluating an automatic defect detector for tealight candles. I am sending you several pictures. Each picture has an ID (for example B01) and three panels of the same tealight:
1. TEST IMAGE: the photograph.
2. REFERENCE: a human-drawn mask; white marks where the real defect is (melted wax around the wick, missing wax, or a folded cup wall).
3. DETECTOR: the detector's anomaly map over the photograph; red and yellow mean the detector thinks the area is defective, blue means normal.

For each picture, score how well the detector's highlighted area matches the reference defect:
5 = the strongest highlights cover the reference defect, with almost nothing highlighted elsewhere.
4 = the defect is mostly covered, with small extra or missing areas.
3 = partly right: only part of the defect is covered, or it is covered but large areas outside it are also highlighted.
2 = the highlights only touch the defect slightly; most of the highlighting is elsewhere.
1 = the highlights miss the defect completely.
Judge only where the highlighted area is and how large it is. Judge each picture on its own.

For each picture, first give one sentence of reasoning. Then finish your answer with one line per picture in exactly this format, and nothing after it:
B01: 4
B02: 2'''
open(f"{UP}/prompt_B.txt", "w").write(PROMPT_B)
print("message 1: B01-B10 + prompt_B.txt")
print("message 2: B11-B20 + prompt_B.txt")

message 1: B01-B10 + prompt_B.txt
message 2: B11-B20 + prompt_B.txt


## Step 1: scores i gave myself


In [7]:
MY_B_SCORES = {
    "B01": 4, "B02": 5, "B03": 4, "B04": 4, "B05": 5,
    "B06": 2, "B07": 4, "B08": 4, "B09": 4, "B10": 5,
    "B11": 4, "B12": 4, "B13": 3, "B14": 2, "B15": 4,
    "B16": 3, "B17": 4, "B18": 3, "B19": 4, "B20": 3,
}
json.dump(MY_B_SCORES, open(f"{OUT_DIR}/my_B_scores.json", "w"), indent=2)
print("filled:", sum(v is not None for v in MY_B_SCORES.values()), "/ 20")

filled: 20 / 20


## Step 2: prompting Gemini


In [8]:
GEMINI_MODEL = "Gemini 3.6 Flash"
GEMINI_DATE = "2026-11-4"

GEMINI_B = r'''
### message 1
Based on the reference defect masks and detector anomaly maps, here is the evaluation:B01: The detector highlights several regions across the candle cup, partially covering the defect but including multiple false positives outside of it.
B02: High-intensity spots touch parts of the folded cup rim, but significant false activations occur along other edges and inside the candle.
B03: The detector highlights multiple circular indentations across the base, covering only parts of the defect while activating over non-defective areas.
B04: The highlight is centered around the wick region matching the reference defect, though a minor secondary region at the bottom edge is also highlighted.
B05: Strong highlights cover key folded regions along the top and bottom edges, leaving central areas with lighter coverage.
B06: The highlight is concentrated near the center wick, missing the extensive top and bottom defect regions identified in the reference.
B07: The detector highlights both the central wick area and the bottom edge, touching only part of the central reference defect.
B08: Strong highlights cover the upper rim folds and indentations, but miss lower portions of the full defect mask.
B09: The detector identifies multiple folded edges and indentations, achieving broad coverage across the defective region with mild over-extension.
B10: The detector highlights the center and lower-right fold areas, touching two defect regions while mostly missing the upper-left defect.   B01: 3
B02: 3
B03: 3
B04: 4
B05: 4
B06: 2
B07: 3
B08: 3
B09: 4
B10: 3

### message 2
For B11, the detector's yellow-red highlights cover almost the entire metal cup area shown in the reference mask, with very little false activation outside.
For B12, the strongest yellow and red highlights fill the entire reference defective area cleanly with minimal extra background detection.
For B13, the detector highlights the center defect region reasonably well, but completely misses the left and right rim defects marked in the reference.
For B14, the central wick area is highlighted, but the large folded rim defects on the left and right sides shown in the reference mask are largely missed.
For B15, the highlighted regions cover almost the full face of the metal cup matching the reference mask closely with minor peripheral variation.
For B16, the main yellow highlight perfectly covers the central melted area with virtually no false positives elsewhere.
For B17, the detector strongly highlights the bottom folded rim defect, but only weakly covers the top folded section and middle center defect.
For B18, the central yellow highlight precisely targets the reference defect at the wick with very low background noise.
For B19, the detector identifies part of the central area and bottom edge, but fails to highlight the prominent lower-left and lower-right rim folds.
For B20, the core yellow activation centers directly over the reference center defect with minimal extraneous highlight.   B11: 5
B12: 5
B13: 2
B14: 2
B15: 4
B16: 5
B17: 3
B18: 5
B19: 2
B20: 5
'''

open(f"{OUT_DIR}/gemini_B_answers.txt", "w").write(GEMINI_B)
json.dump({"model": GEMINI_MODEL, "date": GEMINI_DATE,
           "interface": "Gemini web app, default settings, 10 pictures per message, new chat per message"},
          open(f"{OUT_DIR}/judge_setup.json", "w"), indent=2)

ans_B = {}
for i, v in re.findall(r"\b(B\d{2})\**\s*[:\-]\s*\**\s*([1-5])\b", GEMINI_B, flags=re.I):
    ans_B[i.upper()] = int(v)          # the last line for an ID wins
print("answers:", len(ans_B), "/ 20, missing:", sorted(set(key_B.id) - set(ans_B)))

answers: 20 / 20, missing: []


## Results: can Gemini judge PatchCore's maps the way I do?


In [9]:
B = key_B.copy()
B["name"] = sc.loc[B.row, "name"].values
B["type"] = sc.loc[B.row, "type"].values
B["me"] = B.id.map(MY_B_SCORES).astype(float)
B["gemini"] = B.id.map(ans_B).astype(float)
B["pixel_auroc"] = [roc_auc_score(me[r].ravel(), seg[r].ravel()) for r in B.row]
B.to_csv(f"{OUT_DIR}/exp_B.csv", index=False)

ok = B.dropna(subset=["me", "gemini"])
h, g = ok.me.astype(int), ok.gemini.astype(int)
part2 = {"pictures": len(ok),
         "exact agreement": round((h == g).mean(), 3),
         "within 1 point": round(((h - g).abs() <= 1).mean(), 3),
         "Cohen's kappa": round(cohen_kappa_score(h, g), 3),
         "weighted kappa (quadratic)": round(cohen_kappa_score(h, g, weights="quadratic"), 3),
         "mean score, me": round(h.mean(), 2),
         "mean score, Gemini": round(g.mean(), 2),
         "Spearman, me vs pixel AUROC": round(spearmanr(ok.me, ok.pixel_auroc).correlation, 3),
         "Spearman, Gemini vs pixel AUROC": round(spearmanr(ok.gemini, ok.pixel_auroc).correlation, 3)}
for k, v in part2.items():
    print(f"{k}: {v}")
json.dump(part2, open(f"{OUT_DIR}/exp_B_summary.json", "w"), indent=2)
print(pd.crosstab(h, g, rownames=["me"], colnames=["Gemini"]))
print(B.groupby("type")[["me", "gemini", "pixel_auroc"]].mean().round(2))

pictures: 20
exact agreement: 0.25
within 1 point: 0.7
Cohen's kappa: 0.016
weighted kappa (quadratic): 0.132
mean score, me: 3.75
mean score, Gemini: 3.5
Spearman, me vs pixel AUROC: 0.418
Spearman, Gemini vs pixel AUROC: 0.483
Gemini  2  3  4  5
me                
2       2  0  0  0
3       1  0  0  3
4       1  5  3  2
5       0  2  1  0
         me  gemini  pixel_auroc
type                            
empty  4.22    3.78         0.99
used   3.36    3.27         0.90
